# 06 · Laboratorio supervisado: clasificación multietiqueta de géneros

**Proyecto:** Spotify Music Intelligence
**Módulo 6:** Clasificador multietiqueta (AGENTS.md §16)
**Objetivo:** predecir un conjunto compatible de los 114 géneros a partir de
características acústicas, sin usar el conjunto de test durante la selección.

Unidad de modelado: `recording_group_id`. El split agrupado (70/15/15) está
congelado en `data/processed/splits.parquet`. Este notebook entrena solo los
baselines M0 (frecuencia) y M1 (OneVsRest logistic); los modelos de árboles
M3/M4 se entrenan con `scripts/compare_models.py`.

## Configuración y datos

Se cargan los datos procesados y el dataset multilabel construido por
`spotify_intelligence.classification.datasets`. No se modifica ningún dato.

In [1]:
import os
from pathlib import Path

import pandas as pd

from spotify_intelligence.classification import evaluation as ev
from spotify_intelligence.classification import predict as pr
from spotify_intelligence.classification import thresholds as th
from spotify_intelligence.classification.multilabel import (
    build_model,
    load_model_parameters,
    predict_proba_scores,
)
from spotify_intelligence.classification.training import (
    prepare_base_dataset,
    prepare_training_data,
    split_map_from_dir,
)

cwd = Path.cwd()
if cwd.name == "notebooks":
    os.chdir(cwd.parent)

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)

dataset = prepare_base_dataset("data/processed")
split_map = split_map_from_dir("data/processed")
print("n_samples:", dataset.n_samples)
print("n_labels:", dataset.n_labels)
print(
    "train:",
    len(split_map["train"]),
    "validation:",
    len(split_map["validation"]),
    "test:",
    len(split_map["test"]),
)
print("incomplete:", int(dataset.incomplete_mask.sum()))

n_samples: 83881
n_labels: 114
train: 58716 validation: 12582 test: 12583
incomplete: 145


## Split agrupado y verificación de fuga

Se comprueba que ningún `recording_group_id` aparece en dos conjuntos (§3.5).

In [2]:
from spotify_intelligence.data.splits import verify_disjoint_splits

verify_disjoint_splits(split_map)
print("OK: intersecciones vacías entre train/validation/test")

OK: intersecciones vacías entre train/validation/test


## Baseline M0 · Frecuencia de etiquetas

No usa características; predice la prevalencia observada de cada etiqueta en
train. El umbral global se optimiza en validación (§16.8).

In [3]:
model_params = load_model_parameters("configs/model_parameters.yaml")
data = prepare_training_data(experiment="A")

m0 = build_model("M0", model_params)
m0.fit(data.X_train, data.Y_train)
scores_m0 = predict_proba_scores(m0, data.X_val)
t0 = th.tune_global_threshold(scores_m0, data.Y_val)
print("M0 best threshold:", t0.best_threshold, "score:", round(t0.best_score, 4))

M0 best threshold: 0.1 score: 0.0


In [4]:
pred_m0 = pr.predict_with_threshold(scores_m0, data.dataset.genre_encoder, t0.best_threshold)
ev.evaluate_multilabel(data.Y_val, scores_m0, pred_m0["labels"])

{'macro_f1': 0.0,
 'micro_f1': 0.0,
 'samples_f1': 0.0,
 'hamming_loss': 0.010818783284937924,
 'precision_at_3': 0.029933922458402994,
 'recall_at_5': 0.042538084172476114,
 'hit_at_3': 0.029933922458402994,
 'hit_at_5': 0.052384364302205236,
 'coverage_error': 53.856699307379984,
 'lrap': 0.04721512950362732,
 'per_label_average_precision': [0.010747551946501075,
  0.012658227848101266,
  0.009394156516200939,
  0.0075630921105007565,
  0.011384443913701138,
  0.011862112889101186,
  0.01210094737680121,
  0.011862112889101186,
  0.009473768012100947,
  0.011941724385001195,
  0.0109863864342011,
  0.010349494467001036,
  0.0109863864342011,
  0.011304832417801131,
  0.012737839344001274,
  0.011464055409601147,
  0.00971260249980097,
  0.011543666905501154,
  0.011782501393201179,
  0.006926200143300693,
  0.006289308176100629,
  0.011543666905501154,
  0.0089960990367009,
  0.010827163442401084,
  0.011623278401401163,
  0.011862112889101186,
  0.010269882971101027,
  0.01233978186

## Baseline M1 · One-vs-Rest Logistic Regression

Configuración inicial §16.6: `liblinear`, `C=1.0`, `max_iter=2000`,
`class_weight=balanced`, wrapper `n_jobs=-1`.

In [5]:
m1 = build_model("M1", model_params)
m1.fit(data.X_train, data.Y_train)
scores_m1 = predict_proba_scores(m1, data.X_val)
t1 = th.tune_global_threshold(scores_m1, data.Y_val)
print("M1 best threshold:", t1.best_threshold, "score:", round(t1.best_score, 4))

M1 best threshold: 0.85 score: 0.1386


In [6]:
pred_m1 = pr.predict_with_threshold(scores_m1, data.dataset.genre_encoder, t1.best_threshold)
metrics_m1 = ev.evaluate_multilabel(data.Y_val, scores_m1, pred_m1["labels"])
{k: round(v, 4) for k, v in metrics_m1.items() if k != "per_label_average_precision"}

{'macro_f1': 0.1406,
 'micro_f1': 0.1635,
 'samples_f1': 0.1386,
 'hamming_loss': 0.0354,
 'precision_at_3': 0.3641,
 'recall_at_5': 0.4093,
 'hit_at_3': 0.3641,
 'hit_at_5': 0.4633,
 'coverage_error': 17.0978,
 'lrap': 0.316,
 'label_average_precision_mean': 0.1054,
 'macro_precision': 0.0979,
 'macro_recall': 0.3098}

## Ejemplo de predicción Top-5

La aplicación muestra siempre Top-5. Si ninguna etiqueta supera el umbral, se
muestra el top-1 con un aviso (§16.8).

In [7]:
sample_scores = scores_m1[:3]
topk = pr.top_k_genres(sample_scores, data.dataset.genre_encoder, k=5)
for row in topk:
    print(row)

['idm', 'trip-hop', 'techno', 'afrobeat', 'guitar']
['party', 'ska', 'forro', 'hardcore', 'samba']
['pop-film', 'indian', 'romance', 'jazz', 'malay']


## Limitaciones

- Las puntuaciones no calibradas **no** se llaman probabilidades (§16.10).
- El test congelado se usa solo en la evaluación final autorizada
  (`scripts/evaluate_final_model.py --use-test`).
- La prevalencia real de géneros de Spotify no puede inferirse de esta muestra
  balanceada (§2.3).